# Refresh Availability Signals

Re-seeds the Operations Agent's `ops_availability_signals` table from the current
Gold `stockout_risk` and `demand_forecast` outputs, so the agent keeps seeing
cases after the deploy-time seed ages out of its seven-day window.

## Behavior
- Skips cleanly when the Operations Agent pack is not deployed (no
  `ops_availability_signals` table or no seeded rows).
- Reuses the cohort mode recorded by the deploy (`plan_cohort`): with a native
  Plan the watchlist stays inside the Plan's locked cohort.
- Selects and validates the watchlist exactly like
  `deploy/scripts/_operations_scenario.py`; a repository test asserts both
  produce the same rows.
- Writes nothing when the risk or forecast snapshot is older than seven days,
  because the agent would ignore the rows. The run still succeeds so nightly
  maintenance stays green; readiness reports the stale signals
  (`freshness.operations_signals`) until the stream advances Silver and the
  ML producers rerun.
- The write is tagged with the case set, so a rerun with unchanged inputs does
  not duplicate rows.

## Schedule
Runs in the `daily-maintenance` pipeline after `06-ml-demand-forecast` and
`12-ml-stockout-prediction`.

In [ ]:
import hashlib
import json
import math
import os
import uuid
from datetime import datetime, timedelta, timezone


def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", "retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", "ag")
GOLD_DB = get_env("GOLD_DB", "au")
KQL_DATABASE = get_env("KQL_DATABASE", "retail_eventhouse")
KUSTO_URI = get_env("KUSTO_URI", "")

In [ ]:
# =============================================================================
# SIGNAL SELECTION AND VALIDATION
# =============================================================================
# Mirrors deploy/scripts/_operations_scenario.py. The repository test
# tests/deploy/test_operations_refresh_notebook.py runs this cell and requires
# identical SQL and identical signals for the same inputs.

HIGH_STOCKOUT_PROBABILITY = 0.7
LOW_DAYS_OF_INVENTORY = 3
MAX_SIGNALS = 20
PLAN_STORE_IDS = range(1, 6)
PLAN_PRODUCT_IDS = range(1, 21)
SIGNAL_MAX_AGE_DAYS = 7
SCENARIO = "Protect weekend availability"
PROVENANCE = (
    "Synthetic retail demo; observed from current Gold forecast/risk outputs and Silver "
    "logistics. Inbound units are loads on the road at the snapshot; reservations are "
    "unverified, not assumed available."
)
# Column order of ops_availability_signals (fabric/operations/01-availability-tables.kql).
FIELDS = (
    ("event_ts", "timestamp"),
    ("source_as_of", "timestamp"),
    ("forecast_as_of", "timestamp"),
    ("case_id", "string"),
    ("store_id", "long"),
    ("product_id", "long"),
    ("product_name", "string"),
    ("current_inventory", "long"),
    ("forecast_units_7d", "double"),
    ("stockout_probability", "double"),
    ("risk_horizon_days", "long"),
    ("days_of_inventory", "double"),
    ("shortage_units", "long"),
    ("scenario", "string"),
    ("provenance", "string"),
    ("plan_cohort", "boolean"),
    ("dc_id", "long"),
    ("lead_time_days", "double"),
    ("inbound_units", "long"),
    ("expected_inbound_ts", "timestamp"),
    ("active_incident_id", "string"),
    ("active_incident_type", "string"),
    ("active_incident_cause", "string"),
)


def scenario_query(plan_cohort):
    cohort = (
        f"AND risk.store_id BETWEEN {PLAN_STORE_IDS.start} AND {PLAN_STORE_IDS.stop - 1}\n"
        f"  AND risk.product_id BETWEEN {PLAN_PRODUCT_IDS.start} AND {PLAN_PRODUCT_IDS.stop - 1}"
        if plan_cohort
        else ""
    )
    return f"""
WITH demand AS (
    SELECT store_id, product_id, SUM(predicted_units) AS forecast_units_7d,
           MIN(source_as_of) AS forecast_as_of, COUNT(DISTINCT forecast_date) AS forecast_days,
           COUNT(*) AS forecast_rows, COUNT(DISTINCT source_as_of) AS forecast_versions
    FROM {LAKEHOUSE_NAME}.{GOLD_DB}.demand_forecast
    WHERE forecast_date > CAST(source_as_of AS date)
      AND forecast_date <= date_add(CAST(source_as_of AS date), 7)
    GROUP BY store_id, product_id
),
lanes AS (
    SELECT store_id, MAX(dc_id) AS dc_id
    FROM {LAKEHOUSE_NAME}.{SILVER_DB}.fact_reorders
    GROUP BY store_id
),
trips AS (
    SELECT shipment_id, MAX(store_id) AS store_id,
           MIN(CASE WHEN status = 'SCHEDULED' THEN event_ts END) AS scheduled_ts,
           MIN(CASE WHEN status = 'ARRIVED' THEN event_ts END) AS arrived_ts,
           MIN(planned_arrival_time) AS planned_arrival
    FROM {LAKEHOUSE_NAME}.{SILVER_DB}.fact_truck_moves
    GROUP BY shipment_id
),
lead AS (
    SELECT store_id, AVG((unix_timestamp(arrived_ts) - unix_timestamp(scheduled_ts)) / 86400.0) AS lead_time_days
    FROM trips
    WHERE scheduled_ts IS NOT NULL AND arrived_ts IS NOT NULL
    GROUP BY store_id
),
inbound AS (
    SELECT trips.store_id, cargo.product_id, SUM(cargo.quantity) AS inbound_units,
           MIN(trips.planned_arrival) AS expected_inbound_ts
    FROM {LAKEHOUSE_NAME}.{SILVER_DB}.fact_truck_inventory AS cargo
    JOIN trips ON trips.shipment_id = cargo.shipment_id
    WHERE cargo.action = 'LOAD'
      AND NOT EXISTS (
          SELECT 1 FROM {LAKEHOUSE_NAME}.{SILVER_DB}.fact_truck_inventory AS unloaded
          WHERE unloaded.shipment_id = cargo.shipment_id
            AND unloaded.product_id = cargo.product_id
            AND unloaded.action = 'UNLOAD'
      )
    GROUP BY trips.store_id, cargo.product_id
),
incident AS (
    SELECT snapshot.store_id, link.incident_id, link.incident_type,
           header.description AS incident_cause,
           ROW_NUMBER() OVER (
               PARTITION BY snapshot.store_id ORDER BY link.started_at DESC, link.incident_id
           ) AS incident_rank
    FROM (SELECT DISTINCT store_id, predicted_at FROM {LAKEHOUSE_NAME}.{GOLD_DB}.stockout_risk) AS snapshot
    JOIN {LAKEHOUSE_NAME}.{SILVER_DB}.fact_logistics_incident_stores AS link
      ON link.store_id = snapshot.store_id
     AND link.started_at <= snapshot.predicted_at
     AND (link.resolved_at IS NULL OR link.resolved_at > snapshot.predicted_at)
    JOIN {LAKEHOUSE_NAME}.{SILVER_DB}.fact_logistics_incidents AS header
      ON header.incident_id = link.incident_id
)
SELECT risk.store_id, risk.product_id, product.ProductName AS product_name,
       risk.current_inventory, demand.forecast_units_7d,
       risk.stockout_probability, risk.days_of_inventory,
       risk.forecast_horizon_days AS risk_horizon_days,
       risk.predicted_at AS source_as_of, demand.forecast_as_of, demand.forecast_days,
       demand.forecast_rows, demand.forecast_versions,
       lanes.dc_id, lead.lead_time_days,
       COALESCE(inbound.inbound_units, 0) AS inbound_units, inbound.expected_inbound_ts,
       incident.incident_id AS active_incident_id,
       incident.incident_type AS active_incident_type,
       incident.incident_cause AS active_incident_cause
FROM {LAKEHOUSE_NAME}.{GOLD_DB}.stockout_risk AS risk
JOIN {LAKEHOUSE_NAME}.{SILVER_DB}.dim_products AS product ON product.ID = risk.product_id
JOIN demand ON demand.store_id = risk.store_id AND demand.product_id = risk.product_id
LEFT JOIN lanes ON lanes.store_id = risk.store_id
LEFT JOIN lead ON lead.store_id = risk.store_id
LEFT JOIN inbound ON inbound.store_id = risk.store_id AND inbound.product_id = risk.product_id
LEFT JOIN incident ON incident.store_id = risk.store_id AND incident.incident_rank = 1
WHERE demand.forecast_days = 7 AND demand.forecast_rows = 7 AND demand.forecast_versions = 1
  {cohort}
ORDER BY
    CASE WHEN risk.stockout_probability >= {HIGH_STOCKOUT_PROBABILITY}
           OR risk.days_of_inventory < {LOW_DAYS_OF_INVENTORY} THEN 0 ELSE 1 END,
    risk.stockout_probability DESC, risk.days_of_inventory ASC,
    risk.store_id, risk.product_id
LIMIT {MAX_SIGNALS}
"""


class StaleSnapshotError(ValueError):
    """The Gold snapshot is outside the agent's seven-day window."""


def _utc(value, label):
    if isinstance(value, str):
        value = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if not isinstance(value, datetime):
        raise ValueError(f"{label} is missing.")
    return value.replace(tzinfo=timezone.utc) if value.tzinfo is None else value


def case_identity(store_id, product_id, source_as_of, forecast_as_of):
    return str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"retail-availability:{store_id}:{product_id}:{source_as_of}:{forecast_as_of}",
        )
    )


def meets_alert_rule(signal):
    return (
        signal["stockout_probability"] >= HIGH_STOCKOUT_PROBABILITY
        or signal["days_of_inventory"] < LOW_DAYS_OF_INVENTORY
    )


def supply_context(row):
    """The signal's supply fields (see _operations_scenario.supply_context)."""

    def optional(name, cast):
        value = row.get(name)
        return None if value is None else cast(value)

    expected = row.get("expected_inbound_ts")
    if isinstance(expected, str):
        expected = datetime.fromisoformat(expected.replace("Z", "+00:00"))
    if isinstance(expected, datetime) and expected.tzinfo is None:
        expected = expected.replace(tzinfo=timezone.utc)
    return {
        "dc_id": optional("dc_id", int),
        "lead_time_days": optional("lead_time_days", lambda v: round(float(v), 3)),
        "inbound_units": max(0, int(row.get("inbound_units") or 0)),
        "expected_inbound_ts": (
            expected.astimezone(timezone.utc).isoformat()
            if isinstance(expected, datetime) else None
        ),
        "active_incident_id": optional("active_incident_id", str),
        "active_incident_type": optional("active_incident_type", str),
        "active_incident_cause": optional("active_incident_cause", str),
    }


def build_signals(rows, observed_at, plan_cohort):
    """Validate source rows and build signal rows (see _operations_scenario.scenario_signals)."""
    if not rows or len(rows) > MAX_SIGNALS:
        raise ValueError(
            f"Availability scenario requires 1-{MAX_SIGNALS} real store/product source rows."
        )
    if observed_at.tzinfo is None:
        raise ValueError("Scenario monitoring time must be timezone-aware.")
    signals, keys, snapshot_times = [], set(), []
    for row in rows:
        store_id, product_id = int(row["store_id"]), int(row["product_id"])
        if store_id < 1 or product_id < 1:
            raise ValueError("Availability identifiers must be positive source IDs.")
        if plan_cohort and (
            store_id not in PLAN_STORE_IDS or product_id not in PLAN_PRODUCT_IDS
        ):
            raise ValueError("Availability identifiers are outside the fixed Plan cohort.")
        if (store_id, product_id) in keys:
            raise ValueError("Availability source is not unique by store/product.")
        keys.add((store_id, product_id))
        source_time = _utc(row["source_as_of"], "Availability source time")
        snapshot_times.append(source_time)
        if source_time > observed_at:
            raise ValueError(
                f"Availability source for store {store_id} product {product_id} is future-dated."
            )
        forecast_time = row["forecast_as_of"]
        if isinstance(forecast_time, str):
            forecast_time = datetime.fromisoformat(forecast_time.replace("Z", "+00:00"))
        if (
            not isinstance(forecast_time, datetime)
            or row["forecast_days"] != 7
            or row["forecast_rows"] != 7
            or row["forecast_versions"] != 1
        ):
            raise ValueError("Availability forecast must cover seven distinct days.")
        forecast_time = _utc(forecast_time, "Forecast source time")
        if forecast_time > observed_at:
            raise ValueError(
                f"Forecast source for store {store_id} product {product_id} is future-dated."
            )
        if row["current_inventory"] is None:
            raise ValueError("Availability inventory is unknown.")
        inventory = max(0, int(row["current_inventory"]))
        forecast = float(row["forecast_units_7d"])
        probability = float(row["stockout_probability"])
        risk_horizon = int(row["risk_horizon_days"])
        cover = float(row["days_of_inventory"])
        if (
            not all(math.isfinite(value) for value in (forecast, probability, cover))
            or forecast < 0
            or not 0 <= probability <= 1
            or cover < 0
            or risk_horizon < 1
        ):
            raise ValueError("Availability metrics violate the ML contract.")
        signals.append({
            "event_ts": observed_at.astimezone(timezone.utc).isoformat(),
            "source_as_of": None,
            "forecast_as_of": forecast_time.astimezone(timezone.utc).isoformat(),
            "case_id": None,
            "store_id": store_id,
            "product_id": product_id,
            "product_name": str(row["product_name"]),
            "current_inventory": inventory,
            "forecast_units_7d": forecast,
            "stockout_probability": probability,
            "risk_horizon_days": risk_horizon,
            "days_of_inventory": cover,
            "shortage_units": max(0, math.ceil(forecast) - inventory),
            "scenario": SCENARIO,
            "provenance": PROVENANCE,
            "plan_cohort": plan_cohort,
            **supply_context(row),
        })
    snapshot = max(snapshot_times).astimezone(timezone.utc).isoformat()
    for signal in signals:
        signal["source_as_of"] = snapshot
        signal["case_id"] = case_identity(
            signal["store_id"], signal["product_id"], snapshot, signal["forecast_as_of"]
        )
    for label, field in (("Availability", "source_as_of"), ("Forecast", "forecast_as_of")):
        latest = max(datetime.fromisoformat(signal[field]) for signal in signals)
        age_days = (observed_at - latest).total_seconds() / 86400
        if age_days > SIGNAL_MAX_AGE_DAYS:
            raise StaleSnapshotError(
                f"{label} snapshot is stale: newest {field} {latest.isoformat()} is "
                f"{age_days:.1f} days old (limit {SIGNAL_MAX_AGE_DAYS}); refresh inputs "
                "and required ML first."
            )
    return signals


def signal_batch_identity(signals):
    """Stable tag for one watchlist, so a rerun with unchanged inputs is skipped."""
    cases = "\n".join(sorted(signal["case_id"] for signal in signals))
    return f"ops-signals:{hashlib.sha256(cases.encode('utf-8')).hexdigest()}"

In [ ]:
# =============================================================================
# EVENTHOUSE ACCESS (Spark Kusto connector)
# =============================================================================

KUSTO_FORMAT = "com.microsoft.kusto.spark.synapse.datasource"


def resolve_kusto_uri(database_name):
    """Resolve the KQL database Query URI from the current Fabric workspace."""
    import sempy.fabric as fabric

    client = fabric.FabricRestClient()
    workspace_id = fabric.get_notebook_workspace_id()
    response = client.get(f"v1/workspaces/{workspace_id}/kqlDatabases")
    response.raise_for_status()
    databases = response.json().get("value", [])
    for item in databases:
        if item.get("displayName") == database_name:
            uri = (item.get("properties") or {}).get("queryServiceUri")
            if uri:
                return uri
    found = ", ".join(sorted(d.get("displayName", "?") for d in databases)) or "<none>"
    raise ValueError(
        f"Could not find a KQL database named {database_name!r} in this workspace "
        f"(found: {found}). Set KQL_DATABASE or KUSTO_URI."
    )


if not KUSTO_URI:
    KUSTO_URI = resolve_kusto_uri(KQL_DATABASE)
print(f"Eventhouse database {KQL_DATABASE!r} at {KUSTO_URI}")
token = notebookutils.credentials.getToken(KUSTO_URI)  # noqa: F821


def read_kusto(query):
    return (
        spark.read.format(KUSTO_FORMAT)
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KQL_DATABASE)
        .option("accessToken", token)
        .option("kustoQuery", query)
        .load()
    )


def append_kusto(frame, table, identity):
    """Append a frame by column position; the identity tag makes retries idempotent."""
    tag = f"retail-demo-job:{identity}"
    (
        frame.write.format(KUSTO_FORMAT)
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KQL_DATABASE)
        .option("kustoTable", table)
        .option("accessToken", token)
        .option("tableCreateOptions", "FailIfNotExist")
        .option("writeMode", "Transactional")
        .option("requestId", str(uuid.uuid5(uuid.NAMESPACE_URL, tag)))
        .option("ensureNoDuplicatedBlobs", "true")
        .option("sparkIngestionPropertiesJson", json.dumps({
            "flushImmediately": True,
            "ingestByTags": [tag],
            "ingestIfNotExists": [tag],
        }))
        .mode("Append")
        .save()
    )

In [ ]:
# =============================================================================
# REFRESH
# =============================================================================

from pyspark.sql.types import (
    BooleanType, DoubleType, LongType, StringType, StructField, StructType, TimestampType,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")
SPARK_TYPES = {
    "timestamp": TimestampType(), "string": StringType(), "long": LongType(),
    "double": DoubleType(), "boolean": BooleanType(),
}


def recorded_cohort_mode():
    """Return the deploy's plan_cohort, or None when the pack is not deployed."""
    # isfuzzy tolerates a database without the Operations Agent tables.
    rows = read_kusto(
        "union isfuzzy=true "
        "(datatable(event_ts:datetime, plan_cohort:bool)[]), "
        "(ops_availability_signals "
        "| summarize arg_max(event_ts, recorded=column_ifexists('plan_cohort', bool(null))) "
        "| project event_ts, plan_cohort=recorded) "
        "| where isnotnull(event_ts)"
    ).collect()
    if not rows:
        return None
    if rows[0]["plan_cohort"] is None:
        raise RuntimeError(
            "The latest seeded signals do not record plan_cohort; rerun "
            "deploy_operations_planning once so the refresh knows the cohort mode."
        )
    return bool(rows[0]["plan_cohort"])


def refresh(plan_cohort):
    observed_at = datetime.now(timezone.utc)
    rows = [row.asDict() for row in spark.sql(scenario_query(plan_cohort)).collect()]
    try:
        signals = build_signals(rows, observed_at, plan_cohort)
    except StaleSnapshotError as exc:
        return f"WARNING: no signals refreshed. {exc}"
    alertable = sum(1 for signal in signals if meets_alert_rule(signal))
    if alertable == 0:
        print("WARNING: no refreshed case meets an agent rule; the agent has nothing to alert on.")
    schema = StructType([StructField(name, SPARK_TYPES[kind], True) for name, kind in FIELDS])
    records = [
        tuple(
            datetime.fromisoformat(signal[name])
        if kind == "timestamp" and signal[name] is not None
        else signal[name]
            for name, kind in FIELDS
        )
        for signal in signals
    ]
    append_kusto(
        spark.createDataFrame(records, schema), "ops_availability_signals",
        signal_batch_identity(signals),
    )
    case_list = ", ".join(json.dumps(signal["case_id"]) for signal in signals)
    persisted = read_kusto(
        f"ops_availability_signals | where case_id in ({case_list}) | summarize dcount(case_id)"
    ).collect()[0][0]
    if persisted != len(signals):
        raise RuntimeError(
            f"Refreshed {len(signals)} availability cases but {persisted} are readable."
        )
    return (
        f"Refreshed {len(signals)} availability cases ({alertable} meet an agent rule); "
        f"plan_cohort={plan_cohort}, source_as_of={signals[0]['source_as_of']}."
    )


mode = recorded_cohort_mode()
if mode is None:
    print("The Operations Agent pack has not seeded ops_availability_signals; nothing to refresh.")
else:
    print(refresh(mode))